In [ ]:
import json

# DATA_INFO_PATH = "E:/jqxx/course project/MDD/dataset_info.json"
DATA_NAME = "BCIC2A"
DATA_INFO_PATH = f"E:/jqxx/course project/{DATA_NAME}/dataset_info.json"
with open(DATA_INFO_PATH, "r", encoding="utf-8") as f:
    info = json.load(f)

category_list = info["dataset"]["category_list"]
channels = info["dataset"]["channels"]
target_sampling_rate = info["processing"]["target_sampling_rate"]
window_sec = info["processing"]["window_sec"]

print("=== Dataset Intro ===")
print("Categories:", category_list)
print("Channels (count):", len(channels))
print("Channels:", channels)
print("Target Sampling Rate (Hz):", target_sampling_rate)
print("Window Size (sec):", window_sec)


## 1) 设定数据路径并检查数据形状
这一格会选择数据集（`DATA_NAME`），并读取 `train/val/test` 的基本信息，先确认维度是否符合预期。


In [ ]:
import h5py
import numpy as np
# DATA_NAME = "MDD"
DATA_NAME = "BCIC2A"
# DATA_NAME = "CHINESE"
# DATA_NAME = "MDD"
# DATA_NAME = "SEED"
# DATA_NAME = "SLEEP"
INDEX_PATH_TRAIN = f"E:/jqxx/course project/{DATA_NAME}/train.h5"
INDEX_PATH_VAL = f"E:/jqxx/course project/{DATA_NAME}/val.h5"
INDEX_PATH_TEST = f"E:/jqxx/course project/{DATA_NAME}/test_x_only.h5"

with h5py.File(INDEX_PATH_TRAIN, "r") as f:
    print("keys:", list(f.keys()))
    print("x dtype:", f["X"].dtype)
    print("x shape:", f["X"].shape)
    print("y dtype:", f["y"].dtype)
    print("y shape:", f["y"].shape)
    y = f["y"][()]
    print("unique:", np.unique(y))

## 2) 定义模型：SimpleLinear
最简单的线性分类器，方便和更复杂模型对比。


In [ ]:
import torch
import torch.nn as nn

class SimpleLinear(nn.Module):
    def __init__(self, input_channels, time_points, num_classes):
        super(SimpleLinear, self).__init__()
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(input_channels * time_points, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        return self.fc(x)


In [ ]:
class EEGCNN(nn.Module):
    def __init__(self, input_channels, time_points, num_classes):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(input_channels, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.net(x)
        x = self.classifier(x)
        return x

## 3) 定义模型：SimpleMLP
把输入拉平后通过多层全连接网络进行分类，表达能力比线性模型更强。


In [ ]:
class SimpleMLP(nn.Module):
    def __init__(
        self,
        input_channels,
        num_classes,
        time_points=200,
        hidden_dims=(256, 128),
        dropout=0.3
    ):
        super().__init__()

        input_dim = input_channels * time_points

        layers = []
        prev_dim = input_dim

        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = h

        layers.append(nn.Linear(prev_dim, num_classes))

        self.flatten = nn.Flatten()
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        # x: (B, C, T)
        x = self.flatten(x)      # -> (B, C*T)
        logits = self.mlp(x)     # -> (B, num_classes)
        return logits

## 4) 定义模型：EEGNet
EEGNet 的本质是一个“结构上受约束的 CNN”，通过“时间卷积 + 空间卷积（depthwise）+ 可分离卷积（separable）”分阶段提取 EEG 的时域、频域和空间特征。

卷积结构版本，利用时序与通道方向的局部模式，适合 EEG 信号特征提取。常用于作为baseline


In [ ]:
# class EEGNet(nn.Module):  # EEGNet-8,2
#     def __init__(self, chans,time_point=200,f1=8, d=2, pk1=4, pk2=8, dp=0.5, max_norm1=1,norm=torch.nn.Identity()):
#         super(EEGNet, self).__init__()
#         f2 = f1 * d
#         self.block1 = nn.Sequential(
#             nn.Conv2d(1, f1, (1, 64), padding=(0,32), bias=False),
#             nn.BatchNorm2d(f1),
#         )
#         # Spatial Filters
#         self.block2 = nn.Sequential(
#             nn.Conv2d(f1, d * f1, (chans, 1), groups=f1, bias=False),  # Depthwise Conv
#             nn.BatchNorm2d(d * f1),
#             nn.ELU(),
#             nn.AvgPool2d((1, pk1), stride=pk1),
#             nn.Dropout(dp)
#         )
#         self.block3 = nn.Sequential(
#             nn.Conv2d(d * f1, f2, (1, 16), groups=f2, bias=False, padding=(0,8)),  # Separable Conv
#             nn.Conv2d(f2, f2, kernel_size=1, bias=False),  # Pointwise Conv
#             nn.BatchNorm2d(f2),
#             nn.ELU(),
#             nn.AvgPool2d((1, pk2), stride=pk2),
#             nn.Dropout(dp)
#         )

#         self._apply_max_norm(self.block2[0], max_norm1)
#         self.embed_dim = f2 * ((time_point // pk1) // pk2)
#         self.norm=norm


#     def _apply_max_norm(self, layer, max_norm):
#         for name, param in layer.named_parameters():
#             if 'weight' in name:
#                 param.data = torch.renorm(param.data, p=2, dim=0, maxnorm=max_norm)

#     def forward(self, x):
#         self.norm(x)
#         if len(x.shape) == 2:
#             x = x.unsqueeze(dim=1)
#         x = self.block1(x.unsqueeze(dim=1))
#         x = self.block2(x)
#         x = self.block3(x)
#         return x.flatten(start_dim=1)

In [ ]:
class EEGNet(nn.Module):
    def __init__(self, input_channels, time_points, num_classes):
        super().__init__()

        self.firstconv = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=(1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(8)
        )

        self.depthwise = nn.Sequential(
            nn.Conv2d(8, 16, kernel_size=(input_channels, 1), groups=8, bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(0.3)
        )

        self.separable = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=(1, 16), padding=(0, 8), groups=16, bias=False),
            nn.Conv2d(16, 16, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(0.3)
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, input_channels, time_points)
            out = self.separable(self.depthwise(self.firstconv(dummy)))
            flat_dim = out.numel()

        self.classifier = nn.Linear(flat_dim, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.firstconv(x)
        x = self.depthwise(x)
        x = self.separable(x)
        x = x.flatten(start_dim=1)
        return self.classifier(x)

In [ ]:
class ShallowConvNet(nn.Module):
    def __init__(self, input_channels, time_points, num_classes, dropout=0.6):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 40, kernel_size=(1, 25), bias=False),
            nn.Conv2d(40, 40, kernel_size=(input_channels, 1), bias=False),
            nn.BatchNorm2d(40),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 75), stride=(1, 15)),
            nn.Dropout(dropout)
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, input_channels, time_points)
            out = self.features(dummy)
            flat_dim = out.numel()

        self.classifier = nn.Linear(flat_dim, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.features(x)
        x = x.flatten(start_dim=1)
        return self.classifier(x)

## 5) 导入模型：EEGGRU
这里希望同学们自己手搓一个RNN代码，试试RNN的训练效果如何


In [ ]:
from RNN_Exercise import ExerciseEEGSimpleRNN

## 6) 准备 DataLoader 并训练/验证
这一格完成数据加载、损失函数与优化器设置，然后执行训练循环并记录 `train/val` 指标与曲线。


In [ ]:
import h5py
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from TEST_DATASET import TrainDataset, TestDataset


# CHANNELS = 20
# patch_size = 200
with h5py.File(INDEX_PATH_TRAIN, "r") as f:
    x_shape = f["X"].shape
    y = f["y"][()]

CHANNELS = x_shape[1]
TIME_POINTS = x_shape[2]
CLASSES = len(set(y.tolist()))

print("CHANNELS:", CHANNELS)
print("TIME_POINTS:", TIME_POINTS)
print("CLASSES:", CLASSES)
# CLASSES = 2
BATCH_SIZE = 16
EPOCHS = 50
LR = 5e-4

# -------------------------
# 数据
# -------------------------
train_ds = TrainDataset(INDEX_PATH_TRAIN)
val_ds = TrainDataset(INDEX_PATH_VAL)   
test_ds = TestDataset(INDEX_PATH_TEST)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

# -------------------------
# 模型、损失、优化器
# -------------------------
# model = SimpleMLP(
#     input_channels=CHANNELS,
#     time_points=TIME_POINTS,
#     num_classes=CLASSES
# )
model = EEGNet(
    input_channels=CHANNELS,
    time_points=TIME_POINTS,
    num_classes=CLASSES
)
# model = ShallowConvNet(
#     input_channels=CHANNELS,
#     time_points=TIME_POINTS,
#     num_classes=CLASSES,
#     dropout=0.6
# )
# model = SimpleMLP(input_channels=CHANNELS,time_points=200,num_classes=CLASSES)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# -------------------------
# 记录曲线
# -------------------------
train_losses = []
val_losses = []
val_accuracies = []

# -------------------------
# 训练循环
# -------------------------
best_acc = 0.0
best_model_path = f"best_{DATA_NAME}.pth"
for epoch in range(EPOCHS):
    # ===== Train =====
    model.train()
    train_loss_sum = 0.0
    train_num = 0

    for data, label in train_loader:
        data = data + 0.005 * torch.randn_like(data)
        optimizer.zero_grad()

        output = model(data)
        loss = criterion(output, label)

        loss.backward()
        optimizer.step()

        batch_size = label.size(0)
        train_loss_sum += loss.item() * batch_size
        train_num += batch_size

    epoch_train_loss = train_loss_sum / train_num
    train_losses.append(epoch_train_loss)

    # ===== Val =====
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_num = 0

    with torch.no_grad():
        for val_data, val_label in val_loader:
            val_output = model(val_data)
            val_loss = criterion(val_output, val_label)

            batch_size = val_label.size(0)
            val_loss_sum += val_loss.item() * batch_size
            val_num += batch_size

            val_pred = torch.argmax(val_output, dim=1)
            val_correct += (val_pred == val_label).sum().item()

    epoch_val_loss = val_loss_sum / val_num
    epoch_val_acc = val_correct / val_num

    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_acc)

    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {epoch_val_acc:.4f}"
    )
    if epoch_val_acc > best_acc:
        best_acc = epoch_val_acc
        torch.save(model.state_dict(), best_model_path)

# -------------------------
# 输出最终 val accuracy
# -------------------------

    print(f"Saved best model: {best_acc:.4f}")
print("\n" + "-" * 40)
print(f"Final Val Accuracy: {val_accuracies[-1]:.4f}")

# -------------------------
# 画 train / val loss 曲线
# -------------------------
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train / Val Loss Curve")
plt.legend()
plt.show()


# -------------------------
# test 推理示例
# -------------------------
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds = []

with torch.no_grad():
    for test_data in test_loader:
        test_output = model(test_data)
        test_pred = torch.argmax(test_output, dim=1)
        all_preds.extend(test_pred.cpu().tolist())

output_path = f"{DATA_NAME}.txt"

with open(output_path, "w", encoding="utf-8") as f:
    for p in all_preds:
        f.write(f"{int(p)}\n")

print(f"Saved {len(all_preds)} predictions to {output_path}")
# test_data = next(iter(test_loader))
# test_pred = model(test_data)
# print("-" * 40)
# print("Test input shape:", test_data.shape)
# print("Test output shape:", test_pred.shape)

## 7) 生成并保存 test 预测标签
使用当前训练好的 `model` 在测试集推理，把预测类别按“每行一个数字”写入文本文件。


In [ ]:
# -------------------------
# 保存 test 预测标签（每行一个数字）
# -------------------------
model.load_state_dict(torch.load(f"best_{DATA_NAME}.pth"))
model.eval()
output_path = f'{DATA_NAME}/{DATA_NAME}.txt'

all_test_labels = []
with torch.no_grad():
    for test_data in test_loader:  # test_loader 已经是 shuffle=False
        test_output = model(test_data)
        test_pred = torch.argmax(test_output, dim=1)
        all_test_labels.extend(test_pred.cpu().tolist())

with open(output_path, "w", encoding="utf-8") as f:
    for label in all_test_labels:
        f.write(f"{int(label)}\n")

print(f"Saved {len(all_test_labels)} labels to: {output_path}")
